# Phân tích Siêu tham số Cấu trúc Cây — RandomForestClassifier

Khảo sát ảnh hưởng của 4 tham số điều khiển cấu trúc cây lên **accuracy** và **training time**. Mỗi thí nghiệm giữ `n_estimators=100, random_state=42` và chỉ thay đổi **một tham số** duy nhất.

| Tham số | Ý nghĩa | Default |
|---|---|---|
| `max_depth` | Độ sâu tối đa của mỗi cây | `None` (không giới hạn) |
| `max_leaf_nodes` | Số nút lá tối đa | `None` (không giới hạn) |
| `min_samples_split` | Số mẫu tối thiểu tại nút để chia nhánh | `2` |
| `min_samples_leaf` | Số mẫu tối thiểu mà mỗi lá phải có | `1` |

**Biểu đồ đánh đổi**: mỗi chart có 2 trục — trục trái là accuracy (train vs test), trục phải là training time. Khoảng cách giữa train và test accuracy phản ánh mức độ **overfitting**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
REPORT_DIR = '../reports/figures/'

In [ ]:
df = pd.read_csv('../data/processed/merged_student_data.csv')
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]:,} mẫu  |  Test: {X_test.shape[0]:,} mẫu')
print(f'Features: {X.shape[1]}  |  Pass rate (train): {y_train.mean():.2%}')

In [ ]:
def evaluate_param(param_name, param_values):
    results = []
    n = len(param_values)
    for i, val in enumerate(param_values):
        rf = RandomForestClassifier(
            **{param_name: val},
            n_estimators=100,
            random_state=42,
            n_jobs=-1,
        )
        t0 = time.time()
        rf.fit(X_train, y_train)
        elapsed = time.time() - t0
        train_acc = accuracy_score(y_train, rf.predict(X_train))
        test_acc  = accuracy_score(y_test,  rf.predict(X_test))
        results.append({'value': val, 'train_acc': train_acc,
                        'test_acc': test_acc, 'time': elapsed})
        print(f'  [{i+1}/{n}] {param_name}={val!r}  '
              f'train={train_acc:.4f}  test={test_acc:.4f}  t={elapsed:.1f}s')
    return pd.DataFrame(results)


def plot_tradeoff(df_r, param_name, labels=None, title=None):
    fig, ax1 = plt.subplots(figsize=(12, 6))
    x = list(range(len(df_r)))
    xlabels = labels or df_r['value'].astype(str).tolist()

    ax1.plot(x, df_r['train_acc'], 'o-', color='#1565C0', lw=2, ms=7,
             label='Train Accuracy')
    ax1.plot(x, df_r['test_acc'], 's--', color='#2E7D32', lw=2, ms=7,
             label='Test Accuracy')
    ax1.fill_between(x, df_r['test_acc'], df_r['train_acc'],
                     alpha=0.15, color='red', label='Overfitting Gap')
    ax1.set_ylabel('Accuracy', fontsize=12)
    ax1.set_ylim(0.55, 1.02)
    ax1.set_xticks(x)
    ax1.set_xticklabels(xlabels, rotation=30, ha='right')
    ax1.set_xlabel(param_name, fontsize=12)
    ax1.legend(loc='lower right', fontsize=10)
    ax1.grid(alpha=0.3, linestyle='--')

    ax2 = ax1.twinx()
    ax2.bar(x, df_r['time'], alpha=0.25, color='#E65100', width=0.5,
            label='Training Time (s)')
    ax2.set_ylabel('Training Time (s)', color='#E65100', fontsize=12)
    ax2.tick_params(axis='y', labelcolor='#E65100')
    ax2.legend(loc='upper left', fontsize=10)

    best_idx = df_r['test_acc'].idxmax()
    bx = x[best_idx]
    by = df_r['test_acc'].iloc[best_idx]
    ax1.axvline(x=bx, color='green', linestyle=':', lw=1.5, alpha=0.8)
    ax1.annotate(f'Best: {by:.3f}', xy=(bx, by),
                 xytext=(bx + 0.4, by - 0.06), fontsize=9, color='darkgreen',
                 arrowprops=dict(arrowstyle='->', color='darkgreen'))

    plt.title(title or f'RF — Accuracy & Training Time vs {param_name}',
              fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{REPORT_DIR}rf_{param_name}.png', dpi=150, bbox_inches='tight')
    plt.show()

    print()
    print('  {:<18} {:>8} {:>8} {:>10} {:>8}'.format(
        'Giá trị', 'Train', 'Test', 'Overfit Gap', 'Time'))
    print('  ' + '-' * 56)
    for idx, row in df_r.iterrows():
        lbl  = str(row['value'])
        tr   = row['train_acc']
        te   = row['test_acc']
        gap  = tr - te
        t    = row['time']
        mark = ' <-- BEST' if idx == best_idx else ''
        print(f'  {lbl:<18} {tr:>8.4f} {te:>8.4f} {gap:>10.4f} {t:>7.1f}s{mark}')

## 1. `max_depth` — Độ sâu tối đa của cây

**Ý nghĩa**: Giới hạn số tầng (level) mà mỗi cây có thể phân nhánh.

- `max_depth=1`: cây chỉ có 1 tầng (decision stump) — cực kỳ đơn giản, underfitting cao
- `max_depth=None` (default): cây mọc hoàn toàn cho đến khi mọi lá thuần nhất

**Kỳ vọng đánh đổi**:
- Tăng depth → train accuracy tăng (cây ghi nhớ chi tiết hơn)
- Tăng depth → test accuracy tăng rồi **giảm** khi bắt đầu overfitting
- Tăng depth → training time tăng (nhiều phép tính chia nhánh hơn)

**Câu hỏi**: Cây nông hay sâu? Xem gap train–test để tìm điểm cân bằng.

In [ ]:
print('=== Thí nghiệm: max_depth ===')
depth_vals = [1, 2, 3, 5, 7, 10, 15, 20, 30, None]
depth_labels = [str(v) if v is not None else 'None\n(default)' for v in depth_vals]
df_depth = evaluate_param('max_depth', depth_vals)
plot_tradeoff(df_depth, 'max_depth', labels=depth_labels)

## 2. `max_leaf_nodes` — Số nút lá tối đa

**Ý nghĩa**: Giới hạn tổng số lá trong mỗi cây.

- `max_leaf_nodes=2`: cây chỉ có 1 lần chia — tương đương depth=1
- `max_leaf_nodes=None` (default): không giới hạn

**Khác với `max_depth`**: `max_depth=k` giới hạn đều tất cả nhánh; `max_leaf_nodes` dùng **Best-First growth** — ưu tiên mở nhánh có information gain cao nhất trước, nên cây thường hiệu quả hơn ở cùng số lá.

In [ ]:
print('=== Thí nghiệm: max_leaf_nodes ===')
leaf_node_vals = [2, 5, 10, 20, 50, 100, 200, 500, 1000, None]
leaf_node_labels = [str(v) if v is not None else 'None\n(default)' for v in leaf_node_vals]
df_leaf_nodes = evaluate_param('max_leaf_nodes', leaf_node_vals)
plot_tradeoff(df_leaf_nodes, 'max_leaf_nodes', labels=leaf_node_labels)

## 3. `min_samples_split` — Số mẫu tối thiểu để chia nhánh

**Ý nghĩa**: Một nút nội chỉ được chia tiếp nếu có **ít nhất** `min_samples_split` mẫu.

- `min_samples_split=2` (default): bất kỳ nút nào cũng được chia — cây sâu nhất có thể
- Tăng giá trị → cây đơn giản hơn, ít phân nhánh hơn

**Kỳ vọng đánh đổi**:
- Tăng → train accuracy giảm (cây không ghi nhớ chi tiết)
- Tăng → test accuracy có thể tăng nhẹ (giảm overfit) rồi giảm (underfit)
- Tăng → training time giảm (ít node cần tính toán hơn)

In [ ]:
print('=== Thí nghiệm: min_samples_split ===')
split_vals = [2, 5, 10, 20, 50, 100, 200, 500]
df_split = evaluate_param('min_samples_split', split_vals)
plot_tradeoff(df_split, 'min_samples_split')

## 4. `min_samples_leaf` — Số mẫu tối thiểu tại nút lá

**Ý nghĩa**: Mỗi nút lá phải chứa **ít nhất** `min_samples_leaf` mẫu sau khi phân nhánh.

- `min_samples_leaf=1` (default): lá có thể chỉ chứa 1 mẫu — cây rất sâu, dễ overfit
- Tăng giá trị → lá to hơn → ranh giới phân loại mượt mà hơn

**Khác với `min_samples_split`**:
- `min_samples_split` kiểm soát điều kiện tại nút **CHA** (có đủ mẫu để chia không?)
- `min_samples_leaf` kiểm soát điều kiện tại nút **CON** (sau khi chia, mỗi lá có đủ mẫu không?)

`min_samples_leaf` thường **hiệu quả hơn** trong việc kiểm soát overfitting.

In [ ]:
print('=== Thí nghiệm: min_samples_leaf ===')
leaf_vals = [1, 2, 5, 10, 20, 50, 100, 200]
df_leaf = evaluate_param('min_samples_leaf', leaf_vals)
plot_tradeoff(df_leaf, 'min_samples_leaf')

## 5. Tổng kết: Cây nông hay sâu? Khi nào nên dừng chia?

### Nhận xét chung

| Tham số | Tăng → | Ảnh hưởng accuracy | Ảnh hưởng time |
|---|---|---|---|
| `max_depth` | cây sâu hơn | train↑, test↑ rồi ↓ (overfit) | tăng |
| `max_leaf_nodes` | nhiều lá hơn | tương tự max_depth | tăng |
| `min_samples_split` | cây nhỏ hơn | train↓, test ổn định hơn | giảm |
| `min_samples_leaf` | lá to hơn | train↓, test có thể tăng nhẹ | giảm |

### Khi nào nên dừng chia?
Dừng khi **test accuracy không còn cải thiện** nhưng train accuracy vẫn tăng — đó là điểm bắt đầu overfitting. Biểu hiện: **Overfitting Gap** (vùng đỏ) bắt đầu rộng ra đáng kể.

### Cây nông hay sâu?
- **Cây quá nông** → underfitting, cả train lẫn test đều thấp
- **Cây quá sâu** → overfitting, train rất cao nhưng test thấp
- **Sweet spot**: test accuracy đạt đỉnh trước khi gap train–test mở rộng

In [ ]:
# Best test accuracy từ mỗi tham số
best_per_param = {
    'max_depth': df_depth['test_acc'].max(),
    'max_leaf_nodes': df_leaf_nodes['test_acc'].max(),
    'min_samples_split': df_split['test_acc'].max(),
    'min_samples_leaf': df_leaf['test_acc'].max(),
}

# Baseline: RF với toàn bộ default parameters
rf_default = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_default.fit(X_train, y_train)
default_acc = accuracy_score(y_test, rf_default.predict(X_test))

params = list(best_per_param.keys())
values = list(best_per_param.values())
colors = ['#1565C0', '#2E7D32', '#E65100', '#6A1B9A']

plt.figure(figsize=(9, 5))
bars = plt.bar(params, values, color=colors, alpha=0.8, width=0.5)
plt.axhline(y=default_acc, color='red', linestyle='--', lw=1.5,
            label=f'RF (all defaults): {default_acc:.4f}')
plt.ylim(min(values) - 0.01, min(1.0, max(values) + 0.02))
plt.ylabel('Best Test Accuracy')
plt.title('Best Test Accuracy khi tối ưu từng tham số cấu trúc cây')
plt.legend()
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width() / 2, val + 0.001,
             f'{val:.4f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}rf_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print(f'  RF (all defaults) test acc: {default_acc:.4f}')
print()
for p, v in best_per_param.items():
    delta = v - default_acc
    print(f'  {p:<22}: best test acc = {v:.4f}  ({delta:+.4f} vs default)')

In [ ]:
import json
from pathlib import Path

output = {
    'meta': {
        'n_estimators': 100,
        'test_size': 0.2,
        'random_state': 42,
        'n_train': int(X_train.shape[0]),
        'n_test': int(X_test.shape[0]),
        'n_features': int(X.shape[1]),
        'default_test_acc': round(default_acc, 6),
    },
    'experiments': {
        'max_depth':          df_depth.assign(value=df_depth['value'].astype(str)).to_dict(orient='records'),
        'max_leaf_nodes':     df_leaf_nodes.assign(value=df_leaf_nodes['value'].astype(str)).to_dict(orient='records'),
        'min_samples_split':  df_split.to_dict(orient='records'),
        'min_samples_leaf':   df_leaf.to_dict(orient='records'),
    },
    'best_test_acc': {p: round(v, 6) for p, v in best_per_param.items()},
}

out_path = Path('../reports/rf_hyperparams.json')
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(output, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Saved → {out_path.resolve()}')

## 6. Diminishing Returns — Features & Bản ghi

Theo lý thuyết: tăng thêm features hoặc bản ghi đều giúp accuracy tăng, nhưng đến một ngưỡng nhất định sẽ **chậm dần và dừng lại** (plateau). Hai thí nghiệm dưới kiểm chứng điều này trên OULAD.

| Thí nghiệm | Câu hỏi | Cách thực hiện |
|---|---|---|
| **6.1 Features curve** | Thêm feature thứ k có còn giúp ích không? | Thêm dần từng feature theo thứ tự Gini Importance |
| **6.2 Learning curve** | Thu thập thêm dữ liệu có đáng không? | Tăng dần số mẫu train từ 100 → 26,074 |

Vùng cam = **plateau**: accuracy gần như không đổi khi tăng thêm.

In [ ]:
import json
from pathlib import Path

# ═══════════════════════════════════════════════════════════════
# 6.1  SỐ LƯỢNG FEATURES → ACCURACY
# ═══════════════════════════════════════════════════════════════
# Cột trong CSV đã sắp theo Gini Importance giảm dần (từ eda.py)
feature_cols = [c for c in df.columns if c != 'target']

print('=== Thí nghiệm: Số Features vs Accuracy ===')
feat_results = []
for k in range(1, len(feature_cols) + 1):
    cols_k = feature_cols[:k]
    rf_k = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_k.fit(X_train[cols_k], y_train)
    tr = accuracy_score(y_train, rf_k.predict(X_train[cols_k]))
    te = accuracy_score(y_test,  rf_k.predict(X_test[cols_k]))
    feat_results.append({'k': k, 'feature_added': feature_cols[k-1],
                         'train_acc': tr, 'test_acc': te})
    print(f'  top-{k:2d} (+{feature_cols[k-1]:<24})  train={tr:.4f}  test={te:.4f}')
df_feat = pd.DataFrame(feat_results)

# Tìm điểm plateau: lần đầu tiên gain < 0.2% so với feature liền trước
gains = df_feat['test_acc'].diff().fillna(1)
plateau_mask = gains < 0.002
plateau_k = int(df_feat.loc[plateau_mask, 'k'].min()) if plateau_mask.any() else len(feature_cols)

fig, ax = plt.subplots(figsize=(14, 6))
ks = df_feat['k'].tolist()
ax.plot(ks, df_feat['train_acc'], 'o-', color='#1565C0', lw=2, ms=6, label='Train Accuracy')
ax.plot(ks, df_feat['test_acc'],  's--', color='#2E7D32', lw=2, ms=6, label='Test Accuracy')
ax.fill_between(ks, df_feat['test_acc'], df_feat['train_acc'],
                alpha=0.12, color='red', label='Overfitting Gap')
ax.axvspan(plateau_k - 0.5, len(feature_cols) + 0.5, alpha=0.07, color='orange',
           label=f'Plateau (từ feature #{plateau_k})')
ax.axvline(x=plateau_k, color='darkorange', linestyle=':', lw=1.8)
ax.set_xticks(ks)
ax.set_xticklabels([f'{k}\n{feature_cols[k-1]}' for k in ks],
                   fontsize=7.5, rotation=45, ha='right')
ax.set_xlabel('Feature thứ k (thêm tích lũy theo Gini Importance)', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.legend(fontsize=10, loc='lower right')
ax.grid(alpha=0.3, linestyle='--')
plt.title('Accuracy vs Số lượng Features — Diminishing Returns', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}rf_features_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nPlateau bắt đầu tại feature #{plateau_k}: {feature_cols[plateau_k-1]}')
print(f'Gain thêm {len(feature_cols)-plateau_k+1} features cuối: '
      f'{df_feat["test_acc"].iloc[-1] - df_feat["test_acc"].iloc[plateau_k-2]:.4f}')

# ═══════════════════════════════════════════════════════════════
# 6.2  SỐ LƯỢNG BẢN GHI → ACCURACY (LEARNING CURVE)
# ═══════════════════════════════════════════════════════════════
n_total = len(X_train)
sample_sizes = [100, 300, 600, 1000, 2000, 4000, 8000, 13000, 20000, n_total]

print('\n=== Thí nghiệm: Số Bản ghi vs Accuracy (Learning Curve) ===')
samp_results = []
rng = np.random.RandomState(42)
for n in sample_sizes:
    idx = rng.choice(n_total, n, replace=False)
    Xs, ys = X_train.iloc[idx], y_train.iloc[idx]
    rf_n = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_n.fit(Xs, ys)
    tr = accuracy_score(ys, rf_n.predict(Xs))
    te = accuracy_score(y_test, rf_n.predict(X_test))
    samp_results.append({'n': n, 'train_acc': tr, 'test_acc': te})
    print(f'  n={n:6,}  train={tr:.4f}  test={te:.4f}')
df_samp = pd.DataFrame(samp_results)

# Tìm điểm plateau: gain test < 0.5% trong cửa sổ liền trước
gains_s = df_samp['test_acc'].diff().fillna(1)
plateau_n_mask = gains_s < 0.005
plateau_n = int(df_samp.loc[plateau_n_mask, 'n'].iloc[0]) if plateau_n_mask.any() else n_total

fig, ax = plt.subplots(figsize=(11, 5))
ns = df_samp['n'].tolist()
ax.plot(ns, df_samp['train_acc'], 'o-', color='#1565C0', lw=2, ms=7, label='Train Accuracy')
ax.plot(ns, df_samp['test_acc'],  's--', color='#2E7D32', lw=2, ms=7, label='Test Accuracy')
ax.fill_between(ns, df_samp['test_acc'], df_samp['train_acc'],
                alpha=0.12, color='red', label='Overfitting Gap')
ax.axvline(x=plateau_n, color='darkorange', linestyle=':', lw=1.8,
           label=f'Plateau ~{plateau_n:,} mẫu')
ax.axvspan(plateau_n, n_total * 1.02, alpha=0.07, color='orange')
ax.set_xlabel('Số bản ghi training', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_xticks(ns)
ax.set_xticklabels([f'{n:,}' for n in ns], rotation=30, ha='right')
ax.legend(fontsize=10)
ax.grid(alpha=0.3, linestyle='--')
plt.title('Learning Curve — Accuracy vs Số Bản ghi Training', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}rf_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nPlateau bắt đầu tại ~{plateau_n:,} mẫu')
print(f'Gain thêm {n_total - plateau_n:,} mẫu cuối: '
      f'{df_samp["test_acc"].iloc[-1] - df_samp.loc[df_samp["n"]==plateau_n,"test_acc"].values[0]:.4f}')

# ── Cập nhật JSON ────────────────────────────────────────────────
rf_json = Path('../reports/rf_hyperparams.json')
rf_data = json.loads(rf_json.read_text(encoding='utf-8'))
rf_data['features_curve'] = [
    {'k': int(r['k']), 'feature': r['feature_added'],
     'train_acc': round(r['train_acc'], 6), 'test_acc': round(r['test_acc'], 6)}
    for _, r in df_feat.iterrows()
]
rf_data['learning_curve'] = [
    {'n': int(r['n']), 'train_acc': round(r['train_acc'], 6),
     'test_acc': round(r['test_acc'], 6)}
    for _, r in df_samp.iterrows()
]
rf_json.write_text(json.dumps(rf_data, indent=2, ensure_ascii=False), encoding='utf-8')
print('\nJSON updated: features_curve + learning_curve added.')